# Red Gasista España — Análisis con NetworkX

## ¿Qué es NetworkX?
NetworkX es una librería Python para crear y analizar **grafos**.

Un grafo tiene:
- **Nodos** → puntos de la red (terminales, interconexiones)
- **Aristas** → conexiones entre nodos (gasoductos, flujos)
- **Peso** → cuánto gas fluye por cada conexión (GWh/día)

## ¿Por qué usamos un grafo para el gas?
La red gasista ES un grafo:
- Nodos = puntos de entrada, almacenamientos, distribución
- Aristas = gasoductos que los conectan
- Peso = flujo real en GWh/día

Con el grafo podemos calcular:
1. ¿Qué nodo es más crítico? (centralidad)
2. ¿Cuánto gas máximo puede fluir? (max-flow)
3. ¿Qué pasa si se corta un nodo? (simulación)

In [1]:
# ================================================
# Importaciones y datos del notebook anterior
# ================================================

import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import requests
from datetime import date

print(f"NetworkX versión: {nx.__version__}")
print(f"pandas versión:   {pd.__version__}")
print("✓ Librerías cargadas")

NetworkX versión: 3.6.1
pandas versión:   3.0.1
✓ Librerías cargadas


In [2]:
# ================================================
# Recuperamos los datos de Enagás desde ENTSOG
# ================================================
# Repetimos la llamada del notebook anterior
# para tener los datos frescos en este notebook
# ================================================

print("Descargando datos de Enagás desde ENTSOG...")

resp = requests.get(
    "https://transparency.entsog.eu/api/v1/operationalData",
    params={
        "forceDownload": "true",
        "operatorKey":   "ES-TSO-0006",
        "from":          "2026-01-01",
        "to":            "2026-02-28",
        "indicator":     "Physical Flow",
        "periodType":    "day",
        "limit":         1000
    },
    timeout=60
)

datos = resp.json()
df    = pd.DataFrame(datos['operationalData'])

# Limpieza
df['value_num'] = pd.to_numeric(df['value'], errors='coerce')
df['gwh']       = df['value_num'] / 1_000_000
df              = df[df['gwh'] > 0]

print(f"✓ {len(df)} registros con flujo real")
print(f"  Período: {df['periodFrom'].min()[:10]} → {df['periodFrom'].max()[:10]}")
print(f"  Puntos únicos: {df['pointLabel'].nunique()}")

Descargando datos de Enagás desde ENTSOG...
✓ 872 registros con flujo real
  Período: 2026-01-01 → 2026-02-28
  Puntos únicos: 14


## Paso 1: Construir el grafo de la red gasista

Cada punto de entrada es un NODO.
España (la red nacional) es el nodo central.
Los flujos son las ARISTAS con peso en GWh/día.

La estructura es:
ORIGEN (Argelia, Francia, GNL...) → PUNTO ENTRADA → ESPAÑA

In [3]:
# ================================================
# Calculamos flujo medio por punto
# ================================================

excluir = ['Aggregated Production Sites (ES)',
           'BASIC UGS - AASS Unificado',
           'Aggregated Distribution + Final Consumers Exits (ES)']

df_clean = df[~df['pointLabel'].isin(excluir)].copy()

# Flujo medio diario por punto y dirección
flujos = df_clean.groupby(
    ['pointLabel', 'directionKey']
)['gwh'].mean().round(2).reset_index()

# Solo entradas
entradas = flujos[flujos['directionKey'] == 'entry'].copy()
entradas = entradas.sort_values('gwh', ascending=False)

print("=== NODOS DEL GRAFO ===")
print(entradas.to_string(index=False))
print(f"\nTotal: {entradas['gwh'].sum():.1f} GWh/día")

=== NODOS DEL GRAFO ===
  pointLabel directionKey    gwh
     Almería        entry 328.56
      Bilbao        entry 151.92
   Barcelona        entry 117.17
      Huelva        entry 110.30
     Sagunto        entry 110.12
   Cartagena        entry  74.39
    Mugardos        entry  68.68
VIP PIRINEOS        entry  42.04
       Musel        entry  30.84
 VIP IBERICO        entry  12.31

Total: 1046.3 GWh/día
